# Milestone 5 — VLM Membership Inference Attack
## Method 3: Image Perturbation (Wu et al., EACL 2026)

This notebook implements a membership inference attack (MIA) against a fine-tuned
Vision-Language Model (VLM). Given an image-caption pair, we predict whether that
pair was part of the model's training set.

**Approach:** We extract features from both the fine-tuned (target) model and the
pre-trained base model, then train a classifier on the difference in their outputs.
Members tend to have lower loss and higher token probabilities in the fine-tuned model
compared to the base model.

**Features used:**
- Loss, perplexity, and per-token probability statistics (mean, std, min)
- Min-K% probability (most uncertain tokens)
- Token entropy
- Temperature sensitivity (Hu et al. 2025): members have more peaked distributions
- Zlib compression ratio baseline
- Comparative features: difference/ratio between fine-tuned and base model outputs

**New in Milestone 5:**
- Zlib compression ratio baseline
- Temperature sensitivity features (Hu et al. 2025)
- Formatted results table with standalone baselines

**New in Milestone 5, Method 3:**
- Image perturbation features (Wu et al., EACL 2026)


In [1]:
import sys
import os

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/milestone5/'

sys.path.insert(0, SAVE_DIR)

print(f'sys.path after insert: {sys.path}')
if os.path.exists(SAVE_DIR):
    print(f'Contents of {SAVE_DIR}: {os.listdir(SAVE_DIR)}')
else:
    print(f'{SAVE_DIR} does not exist — creating it.')
    os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive
sys.path after insert: ['/content/drive/MyDrive/milestone5/', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
Contents of /content/drive/MyDrive/milestone5/: ['config.py', 'models.py', 'classifier.py', 'batches.py', '__pycache__', 'features.py', 'train.csv', 'val.csv', 'test.csv']


In [2]:
!pip install -q xgboost
!pip install -q num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 6.2 MB/s eta 0:00:00


In [3]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch

from models import load_data, load_model_and_processor, load_base_model_and_processor
from batches import process_dataset_in_batches
from classifier import build_membership_classifier, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Always load splits — needed for labels and test IDs even when using cached features.
splits = load_data()

Device: cuda
Loading dataset: UBC-SLIME/colx585_group_project_data


train.parquet:   0%|          | 0.00/334M [00:00<?, ?B/s]

dev.parquet:   0%|          | 0.00/67.0M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/338M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Train:      6000 samples  (1500 members)
Validation: 1200 samples  (300 members)
Test:       6000 samples


## 1. Extract Features

Run the cells in this section to extract features from scratch.
Skip to **Section 2** if you already have cached CSVs on Drive.

> **Note:** Features must be re-extracted with the new `features.py` to include
> temperature sensitivity (`loss_at_T05`, `loss_at_T20`, `temp_sensitivity`),


In [4]:
# Load fine-tuned (target) and base (reference) models
processor, model = load_model_and_processor(device)
base_processor, base_model = load_base_model_and_processor(device)

Using dtype: torch.bfloat16


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/403 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

You are using a model of type idefics3 to instantiate a model of type smolvlm. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded.


In [5]:
# Code from Claude AI to prevent training progress loss due to runtime disconnection

import os

train_csv_path = f'{SAVE_DIR}train.csv'

# Find where to resume from if a partial file already exists
start_idx = 0
if os.path.exists(train_csv_path):
    existing = pd.read_csv(train_csv_path)
    start_idx = len(existing)
    print(f'Resuming from sample {start_idx}')
else:
    print('Starting fresh')

train_df = splits['train']

for i in range(start_idx, len(train_df), 200):
    chunk_df = train_df.iloc[i:i+200]
    batch_features = process_dataset_in_batches(
        model, processor, chunk_df, has_labels=True, batch_size=200,
        base_model=base_model, base_processor=base_processor
    )
    mode   = 'w' if i == 0 else 'a'
    header = (i == 0)
    batch_features.to_csv(train_csv_path, mode=mode, header=header, index=False)
    print(f'Saved samples {i} to {min(i+200, len(train_df))}')

print('Train features complete.')

Starting fresh
Processing 200 samples in batches of 200...
  Batch 1/1  (200 samples)


Extracting features: 100%|██████████| 200/200 [04:01<00:00,  1.21s/it]


Done. 200 samples processed.
Saved samples 0 to 200
Processing 200 samples in batches of 200...
  Batch 1/1  (200 samples)


Extracting features: 100%|██████████| 200/200 [04:00<00:00,  1.20s/it]


Done. 200 samples processed.
Saved samples 200 to 400
Processing 200 samples in batches of 200...
  Batch 1/1  (200 samples)


...

Done. 200 samples processed.
Saved samples 5400 to 5600
Processing 200 samples in batches of 200...
  Batch 1/1  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:58<00:00,  1.19s/it]


Done. 200 samples processed.
Saved samples 5600 to 5800
Processing 200 samples in batches of 200...
  Batch 1/1  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:58<00:00,  1.19s/it]

Done. 200 samples processed.
Saved samples 5800 to 6000
Train features complete.


In [6]:
val_features = process_dataset_in_batches(
    model, processor, splits['validation'], has_labels=True, batch_size=200,
    base_model=base_model, base_processor=base_processor
)
val_features.to_csv(f'{SAVE_DIR}val.csv', index=False)
print('Val features saved.')

Processing 1200 samples in batches of 200...
  Batch 1/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:58<00:00,  1.19s/it]


  Batch 2/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:58<00:00,  1.19s/it]


  Batch 3/6  (200 samples)


...

  Batch 5/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:59<00:00,  1.20s/it]


  Batch 6/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:59<00:00,  1.20s/it]

Done. 1200 samples processed.
Val features saved.


In [7]:
test_features = process_dataset_in_batches(
    model, processor, splits['test'], has_labels=False, batch_size=200,
    base_model=base_model, base_processor=base_processor
)
test_features.to_csv(f'{SAVE_DIR}test.csv', index=False)
print('Test features saved.')

Processing 6000 samples in batches of 200...
  Batch 1/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:59<00:00,  1.20s/it]


  Batch 2/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:59<00:00,  1.20s/it]


  Batch 3/30  (200 samples)


...

  Batch 29/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:57<00:00,  1.19s/it]


  Batch 30/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [03:57<00:00,  1.19s/it]


Done. 6000 samples processed.
Test features saved.


## 2. Load Cached Features

Run this cell instead of Section 1 when resuming a session.

In [5]:
import pandas as pd
train_features = pd.read_csv(f'{SAVE_DIR}train.csv')
val_features   = pd.read_csv(f'{SAVE_DIR}val.csv')
test_features  = pd.read_csv(f'{SAVE_DIR}test.csv')
print('Features loaded.')
print(f'  train: {train_features.shape}  val: {val_features.shape}  test: {test_features.shape}')
print(f'  Columns: {list(train_features.columns)}')

Features loaded.
  train: (6000, 49)  val: (1200, 49)  test: (6000, 49)
  Columns: ['caption_loss', 'caption_perplexity', 'caption_length', 'mean_token_prob', 'std_token_prob', 'min_token_prob', 'min_k10_prob', 'min_k20_prob', 'mean_entropy', 'loss_at_T025', 'loss_at_T05', 'loss_at_T20', 'loss_at_T40', 'temp_sensitivity', 'temp_sensitivity_ext', 'zlib_entropy', 'zlib_ratio', 'ref_caption_loss', 'ref_caption_perplexity', 'ref_caption_length', 'ref_mean_token_prob', 'ref_std_token_prob', 'ref_min_token_prob', 'ref_min_k10_prob', 'ref_min_k20_prob', 'ref_mean_entropy', 'ref_loss_at_T025', 'ref_loss_at_T05', 'ref_loss_at_T20', 'ref_loss_at_T40', 'ref_temp_sensitivity', 'ref_temp_sensitivity_ext', 'caption_loss_diff', 'loss_ratio', 'perplexity_ratio', 'mean_prob_diff', 'min_k10_prob_diff', 'min_k20_prob_diff', 'mean_entropy_diff', 'temp_sensitivity_diff', 'temp_sensitivity_ext_diff', 'img_sim_blur_r2', 'img_sim_blur_r2_min', 'img_sim_blur_r2_std', 'img_sim_blur_r5', 'img_sim_blur_r5_min', '

## 3. Train Classifiers & Evaluate

Trains standalone baselines (Loss, Zlib Ratio, Min-K%, Min-K%++) plus
Logistic Regression, XGBoost, and a simple average ensemble.
Results are printed as a formatted table.


In [6]:
results = build_membership_classifier(train_features, val_features)

best_name = max(
    (n for n in results if n not in {"Loss", "Zlib Ratio", "Min-K% (k=10%)", "Min-K%++ (k=10%)"}),
    key=lambda x: results[x]['auc']
)
best = results[best_name]
print(f'\nSelected for submission: {best_name}  AUC={best["auc"]:.4f}')

Features: 47  |  Train: 6000  |  Val: 1200

Individual feature AUCs (top 10):
  caption_loss_diff               AUC=0.7299
  perplexity_ratio                AUC=0.7299
  temp_sensitivity_ext_diff       AUC=0.7007
  temp_sensitivity_diff           AUC=0.6906
  ref_min_k10_prob                AUC=0.6879
  ref_loss_at_T05                 AUC=0.6864
  ref_caption_loss                AUC=0.6813
  ref_caption_perplexity          AUC=0.6813
  ref_temp_sensitivity            AUC=0.6806
  loss_ratio                      AUC=0.6793

Training Logistic Regression...
  AUC=0.7804  TPR@FPR=0.1=0.2467

Training XGBoost...
  AUC=0.7798  TPR@FPR=0.1=0.2700
  Top-5: ['perplexity_ratio', 'caption_loss_diff', 'ref_temp_sensitivity', 'ref_loss_at_T05', 'ref_min_k10_prob']

Building ensemble (simple average)...
  AUC=0.7819  TPR@FPR=0.1=0.2533

── Summary ──
Logistic Regression        AUC=0.7804  TPR@FPR=0.1=0.2467
XGBoost                    AUC=0.7798  TPR@FPR=0.1=0.2700
Ensemble                   AUC=0.78

## 4. Generate Submission

In [7]:
classifier_path = f'{SAVE_DIR}classifier.pkl'
with open(classifier_path, 'wb') as f:
    pickle.dump(best, f)
print(f'Classifier saved to {classifier_path}')

submission = generate_submission(
    splits['test'], test_features, best,
    output_path='submission.csv'
)
print(submission.head())

Classifier saved to /content/drive/MyDrive/milestone5/classifier.pkl
Submission saved to submission.csv  (6000 rows)
Score range: [0.0176, 0.8714]
                   id  is_member
0    cauldron_ln_8044   0.479657
1    cauldron_ln_7863   0.718348
2   mix_SI_UT_5400_19   0.134992
3  mix_SI_UT_5400_896   0.470192
4   cauldron_ln_17842   0.083919


In [9]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>